<a href="https://colab.research.google.com/github/Sayak-coder/HateSpeech-detection/blob/main/Subtask_A_%5BDutch%5D_part_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install transformers

In [3]:
import pandas as pd
import numpy as np
import torch

In [4]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

In [6]:
df_Dt = pd.read_csv('/content/StereoQueerEval_NL_training.tsv', sep='\t')
df_Dt.head()

,StereoQueerEval_id,yt_title,yt_description,yt_comment,stereotype,hate_speech,target
0,training_NL_0001,Non-binaire mensen kunnen binnenkort vragen om...,Binnenkort kunnen non-binaire mensen vragen om...,een schande dat ons geld hier naartoe gaat,no,yes_implicit,group_nb
1,training_NL_0002,Non-binaire mensen kunnen binnenkort vragen om...,Binnenkort kunnen non-binaire mensen vragen om...,"Ach shit,...et is idd 1 april natuurlijk😂....",no,yes_implicit,group_nb
2,training_NL_0003,Acceptatie lhbti+ op dieptepunt! ’Social media...,Ooit was Nederland een tolerant land en de hom...,"Nee, dit accepteren we niet.",no,yes_implicit,group_lgbtqia+
3,training_NL_0004,Pride Amsterdam staat onder spanning: ‘Ermee s...,Amsterdam - ooit de gay-capital maar anno nu a...,Op pride en lhbti gemeenschap mogen het zo gek...,yes,yes_explicit,group_lgbtqia+
4,training_NL_0005,Gehandicapte transgender rapper Arjen wil Nede...,Wij vonden een opmerkelijke verhaal in onze ma...,Een travestiet is niet hetzelfde als een trans...,no,no,none


In [7]:
df = df_Dt

In [8]:
df.isnull().sum()

,0
StereoQueerEval_id,0
yt_title,0
yt_description,0
yt_comment,0
stereotype,0
hate_speech,0
target,0


In [9]:
df['stereotype'].value_counts()

,count
stereotype,
no,1370
yes,868


#Experiment 3.A
only comment as input

In [20]:
X = df['yt_comment']
y = df['stereotype'].map({'no':0, 'yes':1})

In [39]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 45)
X_val, X_test, y_val, y_test = train_test_split(X_test, y_test, test_size = 0.5, random_state = 45)

In [27]:
def build_inputs(d):
    comments = list(d['yt_comment'])
    return comments

def encode(d, tokenizer, max_length=256):
    comments = build_inputs(d)
    return tokenizer(comments, truncation=True, max_length=max_length)

In [28]:
class TextDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

###distilbert-multilingual

In [29]:
from transformers import AutoTokenizer

MODEL = f"distilbert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(MODEL)

In [40]:
train_encodings = tokenizer(X_train.tolist(), truncation=True, max_length=256)
val_encodings = tokenizer(X_val.tolist(), truncation=True, max_length=256)
test_encodings  = tokenizer(X_test.tolist(),  truncation=True, max_length=256)

train_dataset = TextDataset(train_encodings, y_train.tolist())
val_dataset = TextDataset(val_encodings, y_val.tolist())
test_dataset  = TextDataset(test_encodings,  y_test.tolist())

In [32]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    warmup_steps=100,
    weight_decay=0.01,
    learning_rate=2e-5,
    logging_steps=10,
    max_grad_norm=1.0
)

In [33]:
from transformers import DistilBertForSequenceClassification

# Correctly load the pre-trained model and configure it in one step
model = DistilBertForSequenceClassification.from_pretrained(
    MODEL,
    num_labels=3,
    ignore_mismatched_sizes=True,
    id2label={0: "no", 1: "yes"},
    label2id={"no": 0, "yes": 1}
).to(device)

model.safetensors: reconstructing file:   0%|          |  0.00B /  542MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-multilingual-cased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [34]:
from transformers import Trainer, DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer_distilbert = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator
)

trainer_distilbert.train()

Step,Training Loss
10,0.659047
20,0.676011
30,0.677904
40,0.651443
50,0.624199
60,0.679892
70,0.679972
80,0.622926
90,0.597142
100,0.622639


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1120, training_loss=0.4521074628191335, metrics={'train_runtime': 178.0181, 'train_samples_per_second': 50.276, 'train_steps_per_second': 6.291, 'total_flos': 253321132095456.0, 'train_loss': 0.4521074628191335, 'epoch': 5.0})

In [35]:
from sklearn.metrics import confusion_matrix, classification_report

out = trainer_distilbert.predict(test_dataset)
pred = out.predictions.argmax(-1)

print(classification_report(out.label_ids, pred, target_names=["no","yes",], digits=3))
print(confusion_matrix(out.label_ids, pred))

              precision    recall  f1-score   support

          no      0.739     0.733     0.736       135
         yes      0.600     0.607     0.603        89

    accuracy                          0.683       224
   macro avg      0.669     0.670     0.670       224
weighted avg      0.684     0.683     0.683       224

[[99 36]
 [35 54]]


###twitter-roberta-base-hate-latest

In [36]:
from transformers import AutoTokenizer

MODEL = f"cardiffnlp/twitter-roberta-base-hate-latest"
tokenizer = AutoTokenizer.from_pretrained(MODEL)

config.json:   0%|          | 0.00/888 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/351 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

In [41]:
train_encodings = tokenizer(X_train.to_list(), truncation=True, max_length=256)
val_encodings = tokenizer(X_val.to_list(), truncation=True, max_length=256)
test_encodings  = tokenizer(X_test.to_list(),  truncation=True, max_length=256)

train_dataset = TextDataset(train_encodings, y_train.to_list())
val_dataset = TextDataset(val_encodings, y_val.to_list())
test_dataset  = TextDataset(test_encodings,  y_test.to_list())

In [42]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_steps=60,
    weight_decay=0.01,
    learning_rate=2e-5,
    logging_steps=10,
    max_grad_norm=1.0
)

In [43]:
from transformers import AutoModelForSequenceClassification

MODEL = "cardiffnlp/twitter-roberta-base-hate-latest"

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL,
    num_labels=3,
    ignore_mismatched_sizes=True,
    id2label={0: "no", 1: "yes"},
    label2id={"no": 0, "yes": 1}
).to(device)

[transformers] You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  499MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  499MB            

model.safetensors: downloading bytes:           |  0.00B            

In [44]:
from transformers import Trainer, DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer_twitter_roberta = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator
)

trainer_twitter_roberta.train()

Step,Training Loss
10,1.814273
20,1.373412
30,0.671425
40,0.650352
50,0.615197
60,0.602301
70,0.635400
80,0.641435
90,0.687844
100,0.672928


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=560, training_loss=0.5342654743364879, metrics={'train_runtime': 315.5708, 'train_samples_per_second': 28.361, 'train_steps_per_second': 1.775, 'total_flos': 827460630223320.0, 'train_loss': 0.5342654743364879, 'epoch': 5.0})

In [45]:
from sklearn.metrics import confusion_matrix, classification_report

out = trainer_twitter_roberta.predict(test_dataset)
pred = out.predictions.argmax(-1)

print(classification_report(out.label_ids, pred, target_names=["no","yes",], digits=3))
print(confusion_matrix(out.label_ids, pred))

              precision    recall  f1-score   support

          no      0.727     0.711     0.719       135
         yes      0.576     0.596     0.586        89

    accuracy                          0.665       224
   macro avg      0.652     0.653     0.652       224
weighted avg      0.667     0.665     0.666       224

[[96 39]
 [36 53]]


#Experiment 3.B
comment + title as input

In [46]:
df.columns

Index(['StereoQueerEval_id', 'yt_title', 'yt_description', 'yt_comment',
       'stereotype', 'hate_speech', 'target'],
      dtype='object')

In [62]:
X = (df.drop(['StereoQueerEval_id', 'yt_description',
       'stereotype', 'hate_speech', 'target'], axis =1))
y = df['stereotype'].map({'no': 0, 'yes': 1}).to_list()

In [63]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 45)
X_val, X_test, y_val, y_test = train_test_split(X_test, y_test, test_size = 0.5, random_state = 45)

In [64]:
def build_inputs(d):
    comments = d['yt_comment'].tolist()
    title = d['yt_title'].fillna("")
    ctx = title
    return comments, ctx.tolist()

def encode(d, tokenizer, max_length=256):
    comments, ctx = build_inputs(d)
    return tokenizer(comments, ctx, truncation="longest_first", max_length=max_length)

In [65]:
class TextDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

###twitter-roberta-base-hate-latest

In [66]:
from transformers import AutoTokenizer

MODEL = f"cardiffnlp/twitter-roberta-base-hate-latest"
tokenizer = AutoTokenizer.from_pretrained(MODEL)

In [68]:
train_encodings = encode(X_train, tokenizer)
val_encodings = encode(X_val, tokenizer)
test_encodings  = encode(X_test,  tokenizer)

train_dataset = TextDataset(train_encodings, y_train)
val_dataset = TextDataset(val_encodings, y_val)
test_dataset  = TextDataset(test_encodings,  y_test)

In [69]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_steps=60,
    weight_decay=0.01,
    learning_rate=2e-5,
    logging_steps=10,
    max_grad_norm=1.0
)

In [70]:
from transformers import AutoModelForSequenceClassification

MODEL = "cardiffnlp/twitter-roberta-base-hate-latest"

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL,
    num_labels=3,
    ignore_mismatched_sizes=True,
    id2label={0: "no", 1: "yes"},
    label2id={"no": 0, "yes": 1}
).to(device)

[transformers] You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [71]:
from transformers import Trainer, DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer_twitter_roberta = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator
)

trainer_twitter_roberta.train()

Step,Training Loss
10,1.748609
20,1.351596
30,0.708689
40,0.659275
50,0.619484
60,0.607917
70,0.642072
80,0.636666
90,0.688101
100,0.661506


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=560, training_loss=0.5372598737478256, metrics={'train_runtime': 371.7099, 'train_samples_per_second': 24.078, 'train_steps_per_second': 1.507, 'total_flos': 906457669040040.0, 'train_loss': 0.5372598737478256, 'epoch': 5.0})

In [72]:
from sklearn.metrics import confusion_matrix, classification_report

out = trainer_twitter_roberta.predict(test_dataset)
pred = out.predictions.argmax(-1)

print(classification_report(out.label_ids, pred, target_names=["no","yes",], digits=3))
print(confusion_matrix(out.label_ids, pred))

              precision    recall  f1-score   support

          no      0.719     0.778     0.747       135
         yes      0.615     0.539     0.575        89

    accuracy                          0.683       224
   macro avg      0.667     0.659     0.661       224
weighted avg      0.678     0.683     0.679       224

[[105  30]
 [ 41  48]]


###Distilbert-multilingual

In [73]:
from transformers import AutoTokenizer

MODEL = f"distilbert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(MODEL)

In [74]:
train_encodings = encode(X_train, tokenizer)
val_encodings = encode(X_val, tokenizer)
test_encodings  = encode(X_test,  tokenizer)

train_dataset = TextDataset(train_encodings, y_train)
val_dataset = TextDataset(val_encodings, y_val)
test_dataset  = TextDataset(test_encodings,  y_test)

In [75]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    warmup_steps=100,
    weight_decay=0.01,
    learning_rate=2e-5,
    logging_steps=10,
    max_grad_norm=1.0
)

In [76]:
from transformers import DistilBertForSequenceClassification

# Correctly load the pre-trained model and configure it in one step
model = DistilBertForSequenceClassification.from_pretrained(
    MODEL,
    num_labels=3,
    ignore_mismatched_sizes=True,
    id2label={0: "no", 1: "yes"},
    label2id={"no": 0, "yes": 1}
).to(device)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-multilingual-cased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [77]:
from transformers import Trainer, DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer_distilbert = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator
)

trainer_distilbert.train()

Step,Training Loss
10,0.714906
20,0.695460
30,0.694743
40,0.682471
50,0.635503
60,0.688063
70,0.700236
80,0.634123
90,0.601312
100,0.632790


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1120, training_loss=0.47603783298816, metrics={'train_runtime': 217.825, 'train_samples_per_second': 41.088, 'train_steps_per_second': 5.142, 'total_flos': 293217104375832.0, 'train_loss': 0.47603783298816, 'epoch': 5.0})

In [78]:
from sklearn.metrics import confusion_matrix, classification_report

out = trainer_distilbert.predict(test_dataset)
pred = out.predictions.argmax(-1)

print(classification_report(out.label_ids, pred, target_names=["no","yes",], digits=3))
print(confusion_matrix(out.label_ids, pred))

              precision    recall  f1-score   support

          no      0.782     0.770     0.776       135
         yes      0.659     0.674     0.667        89

    accuracy                          0.732       224
   macro avg      0.721     0.722     0.721       224
weighted avg      0.733     0.732     0.733       224

[[104  31]
 [ 29  60]]


#Experiment 3.C
comment + title + description as input

In [79]:
df.columns

Index(['StereoQueerEval_id', 'yt_title', 'yt_description', 'yt_comment',
       'stereotype', 'hate_speech', 'target'],
      dtype='object')

In [81]:
X = df.drop(['StereoQueerEval_id',
       'stereotype', 'hate_speech', 'target'], axis =1)
y = df['stereotype'].map({'no':0 ,'yes':1}).to_list()

In [83]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 45)
X_val, X_test, y_val, y_test = train_test_split(X_test, y_test, test_size = 0.5, random_state = 45)

In [84]:
def build_inputs(d):
    comments = d['yt_comment'].tolist()
    title = d['yt_title'].fillna("")
    desc  = d['yt_description'].fillna("").str[:300]        # trim long descriptions
    ctx = (title + " . " + desc)
    return comments, ctx.tolist()

def encode(d, tokenizer, max_length=256):
    comments, ctx = build_inputs(d)
    return tokenizer(comments, ctx, truncation="longest_first", max_length=max_length)

In [85]:
class TextDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

###twitter-roberta-base-hate-latest

In [86]:
from transformers import AutoTokenizer

MODEL = f"cardiffnlp/twitter-roberta-base-hate-latest"
tokenizer = AutoTokenizer.from_pretrained(MODEL)

In [88]:
train_encodings = encode(X_train, tokenizer)
val_encodings = encode(X_val, tokenizer)
test_encodings  = encode(X_test,  tokenizer)

train_dataset = TextDataset(train_encodings, y_train)
val_dataset = TextDataset(val_encodings, y_val)
test_dataset  = TextDataset(test_encodings,  y_test)

In [90]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_steps=60,
    weight_decay=0.01,
    learning_rate=2e-5,
    logging_steps=10,
    max_grad_norm=1.0
)

In [91]:
from transformers import AutoModelForSequenceClassification

MODEL = "cardiffnlp/twitter-roberta-base-hate-latest"

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL,
    num_labels=3,
    ignore_mismatched_sizes=True,
    id2label={0: "no", 1: "yes"},
    label2id={"no": 0, "yes": 1}
).to(device)

[transformers] You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [92]:
from transformers import Trainer, DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer_twitter_roberta = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator
)

trainer_twitter_roberta.train()

Step,Training Loss
10,1.772980
20,1.097770
30,0.739471
40,0.699410
50,0.628674
60,0.612863
70,0.641739
80,0.619301
90,0.655725
100,0.653186


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=560, training_loss=0.5602329922573906, metrics={'train_runtime': 444.7434, 'train_samples_per_second': 20.124, 'train_steps_per_second': 1.259, 'total_flos': 1155336060749160.0, 'train_loss': 0.5602329922573906, 'epoch': 5.0})

In [93]:
from sklearn.metrics import confusion_matrix, classification_report

out = trainer_twitter_roberta.predict(test_dataset)
pred = out.predictions.argmax(-1)

print(classification_report(out.label_ids, pred, target_names=["no","yes",], digits=3))
print(confusion_matrix(out.label_ids, pred))

              precision    recall  f1-score   support

          no      0.719     0.741     0.730       135
         yes      0.588     0.562     0.575        89

    accuracy                          0.670       224
   macro avg      0.654     0.651     0.652       224
weighted avg      0.667     0.670     0.668       224

[[100  35]
 [ 39  50]]


###Distilber-multilingual

In [94]:
from transformers import AutoTokenizer

MODEL = f"distilbert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(MODEL)

In [95]:
train_encodings = encode(X_train, tokenizer)
val_encodings = encode(X_val, tokenizer)
test_encodings  = encode(X_test,  tokenizer)

train_dataset = TextDataset(train_encodings, y_train)
val_dataset = TextDataset(val_encodings, y_val)
test_dataset  = TextDataset(test_encodings,  y_test)

In [96]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    warmup_steps=100,
    weight_decay=0.01,
    learning_rate=2e-5,
    logging_steps=10,
    max_grad_norm=1.0
)

In [97]:
from transformers import DistilBertForSequenceClassification

# Correctly load the pre-trained model and configure it in one step
model = DistilBertForSequenceClassification.from_pretrained(
    MODEL,
    num_labels=3,
    ignore_mismatched_sizes=True,
    id2label={0: "no", 1: "yes"},
    label2id={"no": 0, "yes": 1}
).to(device)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-multilingual-cased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [98]:
from transformers import Trainer, DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer_distilbert = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator
)

trainer_distilbert.train()

Step,Training Loss
10,0.718283
20,0.702543
30,0.692729
40,0.672078
50,0.641162
60,0.693487
70,0.715406
80,0.640950
90,0.617994
100,0.634953


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1120, training_loss=0.5731809547969273, metrics={'train_runtime': 287.8994, 'train_samples_per_second': 31.087, 'train_steps_per_second': 3.89, 'total_flos': 463361128933944.0, 'train_loss': 0.5731809547969273, 'epoch': 5.0})

In [99]:
from sklearn.metrics import confusion_matrix, classification_report

out = trainer_distilbert.predict(test_dataset)
pred = out.predictions.argmax(-1)

print(classification_report(out.label_ids, pred, target_names=["no","yes",], digits=3))
print(confusion_matrix(out.label_ids, pred))

              precision    recall  f1-score   support

          no      0.756     0.689     0.721       135
         yes      0.584     0.663     0.621        89

    accuracy                          0.679       224
   macro avg      0.670     0.676     0.671       224
weighted avg      0.688     0.679     0.681       224

[[93 42]
 [30 59]]
